In [4]:
# ============================
# 环境准备与数据加载
# ============================
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

# 加载 Titanic 数据
df = pd.read_csv('titanic.csv')
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'
df_model = df[features + [target]].copy()
X = df_model.drop(columns=target)
y = df_model[target]

# 划分训练集和测试集（保持与原 notebook 一致的分层划分）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"训练集大小: {X_train.shape}, 测试集大小: {X_test.shape}")



训练集大小: (712, 7), 测试集大小: (179, 7)


In [5]:
# ==================================================
# 练习 1：添加 LogTransformer 对 fare 列应用 log1p
# ==================================================
class LogTransformer(BaseEstimator, TransformerMixin):
    """对指定列应用 log1p 变换"""
    def __init__(self, columns=None):
        self.columns = columns
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[col] = np.log1p(X[col])
        return X

# 原管道（无 log1p）
preprocessor_original = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), ['age', 'fare', 'sibsp', 'parch']),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['sex', 'embarked']),
        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder()),
        ]), ['pclass']),
    ],
    remainder='drop'
)
pipe_original = Pipeline([
    ('pre', preprocessor_original),
    ('model', RandomForestClassifier(random_state=42, n_estimators=100))
])

# 新管道：在 ColumnTransformer 之前对 fare 做 log1p
preprocessor_log = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), ['age', 'sibsp', 'parch']),   # fare 已单独处理，不再放这里
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['sex', 'embarked']),
        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder()),
        ]), ['pclass']),
    ],
    remainder='drop'
)
pipe_log = Pipeline([
    ('log_fare', LogTransformer(columns=['fare'])),
    ('pre', preprocessor_log),
    ('model', RandomForestClassifier(random_state=42, n_estimators=100))
])

cv_original = cross_val_score(pipe_original, X_train, y_train, cv=5, scoring='accuracy')
cv_log = cross_val_score(pipe_log, X_train, y_train, cv=5, scoring='accuracy')

print("\n========== 练习 1 结果 ==========")
print(f"原管道 CV 准确率: {cv_original.mean():.4f} ± {cv_original.std():.4f}")
print(f"加入 fare log1p 管道 CV 准确率: {cv_log.mean():.4f} ± {cv_log.std():.4f}")
print(f"提升: {cv_log.mean() - cv_original.mean():.4f}")

# ==================================================
# 练习 2：从 cabin 提取 deck 列
# ==================================================
# 重新加载数据，包含 cabin 列
df_full = sns.load_dataset('titanic')
X_full = df_full.drop(columns='survived')
y_full = df_full['survived']
X_train_deck, X_test_deck, y_train_deck, y_test_deck = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=42
)

class DeckExtractor(BaseEstimator, TransformerMixin):
    """从 cabin 列提取 deck（第一个字母），缺失则填充 'Unknown'"""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        deck = X['cabin'].str[0].fillna('Unknown')
        return deck.values.reshape(-1, 1)

preprocessor_deck = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), ['age', 'fare', 'sibsp', 'parch']),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['sex', 'embarked']),
        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder()),
        ]), ['pclass']),
        ('deck', Pipeline([
            ('extract', DeckExtractor()),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['cabin']),
    ],
    remainder='drop'
)
pipe_deck = Pipeline([
    ('pre', preprocessor_deck),
    ('model', RandomForestClassifier(random_state=42, n_estimators=100))
])

cv_deck = cross_val_score(pipe_deck, X_train_deck, y_train_deck, cv=5, scoring='accuracy')
print("\n========== 练习 2 结果 ==========")
print(f"含 deck 的管道 CV 准确率: {cv_deck.mean():.4f} ± {cv_deck.std():.4f}")

# ==================================================
# 练习 3：GridSearchCV 调优 age 插补策略和 RandomForest 参数
# ==================================================
# 注意：这里使用最初的 X_train（不含 cabin），方便比较
preprocessor_tune = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer()),   # strategy 待搜索
            ('scale', StandardScaler()),
        ]), ['age', 'fare', 'sibsp', 'parch']),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), ['sex', 'embarked']),
        ('ord', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder()),
        ]), ['pclass']),
    ],
    remainder='drop'
)

pipe_tune = Pipeline([
    ('pre', preprocessor_tune),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'pre__num__impute__strategy': ['median', 'most_frequent'],
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 8, None]
}

grid_search = GridSearchCV(pipe_tune, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("\n========== 练习 3 结果 ==========")
print("最佳参数:", grid_search.best_params_)
print(f"最佳 CV 准确率: {grid_search.best_score_:.4f}")
print(f"测试集准确率: {grid_search.score(X_test, y_test):.4f}")

# ==================================================
# 练习 4：演示外部拟合 scaler 导致分数膨胀
# ==================================================
# 错误做法：在整个 X_train 上拟合 scaler，再交叉验证
scaler = StandardScaler()
X_train_scaled_wrong = scaler.fit_transform(X_train[['age', 'fare']])
X_train_rest = X_train[['sibsp', 'parch', 'pclass']].values
X_train_wrong = np.hstack([X_train_scaled_wrong, X_train_rest])
model_lr = LogisticRegression(max_iter=1000, random_state=42)
scores_wrong = cross_val_score(model_lr, X_train_wrong, y_train, cv=5, scoring='accuracy')

# 正确做法：将 scaler 放在 Pipeline 中
pipe_correct = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([('scale', StandardScaler())]), ['age', 'fare']),
    ], remainder='passthrough')),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
X_train_correct = X_train[['age', 'fare', 'sibsp', 'parch', 'pclass']]
scores_correct = cross_val_score(pipe_correct, X_train_correct, y_train, cv=5, scoring='accuracy')

print("\n========== 练习 4 结果 ==========")
print(f"错误方式（外部拟合）CV 准确率: {scores_wrong.mean():.4f} ± {scores_wrong.std():.4f}")
print(f"正确方式（Pipeline内）CV 准确率: {scores_correct.mean():.4f} ± {scores_correct.std():.4f}")
print(f"膨胀差值: {scores_wrong.mean() - scores_correct.mean():.4f}")


========== 练习 1 结果 ==========
原管道 CV 准确率: 0.7895 ± 0.0528
加入 fare log1p 管道 CV 准确率: 0.7922 ± 0.0280
提升: 0.0027


URLError: <urlopen error [WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。>